# Khmer OCR — Predict a folder with `vgg_bilstm_ctc_10k`

Uses this repo's `final/predict.py`, which rebuilds the model from its own
`experiment.json` config and loads the charset it was trained on — the right tool
for runs under `outputs_crnn/`.

**How to use:** drop images into the `images/` folder next to this notebook, then run
the cell below. Every image (png/jpg/jpeg/bmp/tif/webp, searched recursively) gets
predicted, printed inline, and saved to `output.txt`.

In [ ]:
import sys
from pathlib import Path

import torch

# ===================== EDIT THESE =====================
IMAGES_DIR = "images"               # drop your images here, then run this cell
RUN_NAME   = "vgg_bilstm_ctc_10k"   # which trained run under outputs_crnn/ to use
# ======================================================

# Make `final/predict.py` importable whether the notebook runs from final/ or repo root.
FINAL_DIR = Path.cwd() if (Path.cwd() / "predict.py").exists() else Path.cwd() / "final"
sys.path.insert(0, str(FINAL_DIR))
import predict as P

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Build just the one run (it rebuilds from experiment.json + its own training charset).
predictors = P.load_predictors(
    P.DEFAULT_OUTPUTS_ROOT, P.DEFAULT_CHARSET, device, only=[RUN_NAME]
)
assert predictors, f"Run '{RUN_NAME}' not found under {P.DEFAULT_OUTPUTS_ROOT}"
predictor = predictors[0]
print(f"Using {predictor.run_name} ({predictor.arch})\n")

# Predict every image in the folder.
Path(IMAGES_DIR).mkdir(parents=True, exist_ok=True)
images = P.gather_images([IMAGES_DIR])
print(f"Found {len(images)} image(s) in '{IMAGES_DIR}'\n")

lines = []
for img in images:
    try:
        text = predictor.predict(img)
    except Exception as e:
        text = f"<error: {type(e).__name__}: {e}>"
    line = f"{img.name}\n  -> {text}"
    print(line + "\n")
    lines.append(line)

Path("output.txt").write_text("\n".join(lines), encoding="utf-8")
print("Saved -> output.txt")